In [8]:
import os
from pyspark.sql import SparkSession
from pyspark.sql.functions import explode, col, from_json
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, ArrayType, IntegerType

In [9]:
# Configura Spark para MinIO / S3

os.environ["PYSPARK_SUBMIT_ARGS"] = (
    "--packages org.apache.hadoop:hadoop-aws:3.3.2,"
    "com.amazonaws:aws-java-sdk-bundle:1.12.180 pyspark-shell"
)

spark = SparkSession.builder \
    .appName("SparkMinIO") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.hadoop.fs.s3a.access.key", "Projeto_Final") \
    .config("spark.hadoop.fs.s3a.secret.key", "Projeto_Final") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .getOrCreate()

In [10]:
veiculo_schema = StructType([
    StructField("p", StringType(), True),
    StructField("a", StringType(), True),
    StructField("ta", StringType(), True),
    StructField("px", DoubleType(), True),
    StructField("py", DoubleType(), True)
])

linha_schema = StructType([
    StructField("c", StringType(), True),
    StructField("cl", StringType(), True),
    StructField("lt0", StringType(), True),
    StructField("lt1", StringType(), True),
    StructField("sl", IntegerType(), True),
    StructField("vs", ArrayType(veiculo_schema), True)
])

posicao_schema = StructType([
    StructField("hr", StringType(), True),
    StructField("l", ArrayType(linha_schema), True)
])

In [11]:
# Streaming da camada raw
df_bronze_stream = spark.readStream \
    .format("json") \
    .option("maxFilesPerTrigger", 1) \
    .schema(posicao_schema) \
    .load("s3a://raw/sptrans/posicoes/")

In [13]:
# Ignora arquivos sem conteúdo
from pyspark.sql.functions import col, explode, size
df_bronze_stream = df_bronze_stream.filter(size(col("l")) > 0)

In [14]:
# Explode e transforma como no batch
def transform_batch(batch_df, batch_id):
    df_linhas = batch_df.select(
        col("hr"),
        explode(col("l")).alias("linha")
    )

    df_linhas_selecionadas = df_linhas.select(
        "hr",
        col("linha.c").alias("codigo_linha"),
        col("linha.cl").alias("id_linha"),
        col("linha.lt0").alias("terminal_origem"),
        col("linha.lt1").alias("terminal_destino"),
        col("linha.sl").alias("sentido"),
        explode(col("linha.vs")).alias("veiculo")
    )

    df_veiculos = df_linhas_selecionadas.select(
        "hr",
        "codigo_linha",
        "id_linha",
        "terminal_origem",
        "terminal_destino",
        "sentido",
        col("veiculo.p").alias("prefixo"),
        col("veiculo.a").alias("operando"),
        col("veiculo.ta").alias("atualizacao"),
        col("veiculo.px").alias("longitude"),
        col("veiculo.py").alias("latitude")
    )
    
    # Só salva se o batch tiver mais de X linhas
    if df_veiculos.count() > 5:  # ajuste o valor conforme necessário
        df_veiculos.write \
            .mode("append") \
            .parquet("s3a://silver/posicoes")
    else:
        print(f"Batch {batch_id} ignorado: poucos registros ({df_veiculos.count()})")


In [ ]:
# Inicia o streaming usando foreachBatch
query = df_bronze_stream.writeStream \
    .foreachBatch(transform_batch) \
    .option("checkpointLocation", "s3a://silver/checkpoints/posicoes") \
    .start()

query.awaitTermination()